Input Boss Name

Get boss's data from db

seperate data into decks with 6+ wins, and decks with less

Create a dictionary for each set of performers

sum up all the cards from each deck into their respective dicts

Take the average of everything

In [1]:

from collections import defaultdict

import card_data_collection_pipeline
import db_operations
import helpers

import pandas as pd


In [2]:

def db_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return db_operations.find_first_in_table('main_table', 'card_data', {'id':id})


def url_lookup(id_and_name):
    id = helpers.extract_card_id(id_and_name)
    return card_data_collection_pipeline.add_card_info_to_db(id)
    print(id)
    raise NotImplementedError 


In [3]:
BOSS_NAME = "Deeplands, Reguregnus"

In [4]:
full_df = db_operations.get_answer_from_table(database='main_table',
                                              table='all_events',
                                              query={'boss':
                                                     {'$regex':BOSS_NAME}
                                                     }
                                             )

In [5]:
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,regalis_piece,crit_heal_count,draw_count,stand_heal_count
0,69b04a223efa47b42b6c3cf0,9,Accel,"Deeplands, Reguregnus",6,Stoicheia,6FZCT,"{'RideDeck': {'DZ-BT12/SR32EN : Deeplands, Reg...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,2.0
1,69b04a223efa47b42b6c3cf6,15,Rob,"Deeplands, Reguregnus",5,Stoicheia,3ACHV,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Fire Regalis,0.0,1.0,2.0
2,69b04a223efa47b42b6c3d07,32,Willow,"Deeplands, Reguregnus",5,Stoicheia,7PBFV,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,1.0
3,69b04a223efa47b42b6c3d2d,70,Dua Dharma,"Deeplands, Reguregnus",4,Stoicheia,5AFW3,"{'RideDeck': {'DZ-BT12/FFR14EN : Deeplands, Re...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,1.0
4,69b04a223efa47b42b6c3d4a,99,Zessy,"Deeplands, Reguregnus",3,Stoicheia,1J6R5,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,1.0,1.0,1.0


Currently, I'm noticing two bugs with the system.
One: Alphnights keeps getting added to the db again and again, so it's failing to read??
Also noticed maybe the same thing with hakasui and remred?

Two: Most cards with no flavor text have an issue with the format~artist 'line'

In [6]:
# lets try just saving and re loading one copy of alphknights. If we can do one, we can do more :)
# memoizer = memoizer or None
# memoizer

In [7]:
memoizer = dict()

In [8]:
# memoizer = memoizer or dict()
copy = full_df
for i, row in full_df.iterrows():
    new_deck = defaultdict(int)
    for card_id_and_name, amount in row.deck['MainDeck'].items():
        # print(card_id_and_name,'\n', amount,'\n~~~~~~~~~~~~~~~~~~~~~~')
        # if type(amount) != type(67):
        #     print(type(amount))
        id = helpers.extract_card_id(card_id_and_name)
        try:
            if memoizer[id]:
                lookup = memoizer[id]
        except KeyError:
            lookup = None

        if not lookup:
            lookup = db_lookup(card_id_and_name)
            memoizer[id] = lookup
        if not lookup:
            lookup = url_lookup( card_id_and_name)
            memoizer[id] = lookup

        name = lookup['name']

        if 'Trigger' in lookup['type']:
            if 5000 == lookup['power']:
                name = 'Vanilla Trigger'

        if '[CONT]:Sentinel' in lookup['effect']:
            if 'Unit' in lookup['type']:
                name = 'Sentinel'
            # else:
            #     name = 'Elementaria'

        if '[CONT]:This card is' in lookup['effect']:
            first = lookup['effect'].find('"')+1
            second = lookup['effect'].find('"', first)
            name = lookup['effect'][first:second]

        new_deck[name] += amount
    
    copy.at[i, 'deck']['MainDeck'] = new_deck

In [ ]:
# foo = card_data_collection_pipeline.get_card_info_from_bushi_site('DZ-SS08/007EN')
# foo

In [9]:
full_df.head()

,_id,rank,name,boss,wins,nation,decklog,deck,location,region,date,regalis_piece,crit_heal_count,draw_count,stand_heal_count
0,69b04a223efa47b42b6c3cf0,9,Accel,"Deeplands, Reguregnus",6,Stoicheia,6FZCT,"{'RideDeck': {'DZ-BT12/SR32EN : Deeplands, Reg...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,2.0
1,69b04a223efa47b42b6c3cf6,15,Rob,"Deeplands, Reguregnus",5,Stoicheia,3ACHV,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Fire Regalis,0.0,1.0,2.0
2,69b04a223efa47b42b6c3d07,32,Willow,"Deeplands, Reguregnus",5,Stoicheia,7PBFV,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,1.0
3,69b04a223efa47b42b6c3d2d,70,Dua Dharma,"Deeplands, Reguregnus",4,Stoicheia,5AFW3,"{'RideDeck': {'DZ-BT12/FFR14EN : Deeplands, Re...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,0.0,1.0,1.0
4,69b04a223efa47b42b6c3d4a,99,Zessy,"Deeplands, Reguregnus",3,Stoicheia,1J6R5,"{'RideDeck': {'DZ-BT12/014EN : Deeplands, Regu...","Melbourne, Australia",AO,"February 28, 2026",Protection: Twincast,1.0,1.0,1.0


In [10]:
WINS = 5
top_df = full_df[full_df['wins'] > WINS]
bot_df = full_df[full_df['wins'] <= WINS]

In [11]:

top_dict = defaultdict(int)
bot_dict = defaultdict(int)
all_dict = defaultdict(int)

for dict in top_df.deck:
    for k, v in dict['MainDeck'].items():
        top_dict[k] += v
        all_dict[k] += v

for dict in bot_df.deck:
    for k, v in dict['MainDeck'].items():
        bot_dict[k] += v
        all_dict[k] += v
        

In [12]:
num_tops = len(top_df)
num_bots = len(bot_df)
num_total = num_tops + num_bots

print("Number of Tops: ", num_tops)
print("Number of Bots: ", num_bots)
print("Total Number: ", num_total)

Number of Tops:  5
Number of Bots:  22
Total Number:  27


In [13]:
top_dict = {k: v / num_tops for k,v in top_dict.items()}
bot_dict = {k: v / num_bots for k,v in bot_dict.items()}
all_dict = {k: v / num_total for k,v in all_dict.items()}

top_dict = {k:v for k,v in sorted(top_dict.items(), key=lambda k: k[1], reverse=True)}
bot_dict = {k:v for k,v in sorted(bot_dict.items(), key=lambda k: k[1], reverse=True)}
all_dict = {k:v for k,v in sorted(all_dict.items(), key=lambda k: k[1], reverse=True)}

In [14]:
top_dict

{'Vanilla Trigger': 8.6,
 'Deeplands, Reguregnus Athanasios': 4.0,
 'Call of Deep Lands': 4.0,
 'Harvest Elf': 3.8,
 'Deeplands, Orgady': 3.4,
 'Sentinel': 3.4,
 'Aspiring Maiden, Alana': 3.2,
 'Deeplands, Reguregnus': 3.0,
 'Deeplands, Meydith': 3.0,
 'Conceited Noble, Philander': 3.0,
 'Rotting Usurp Dragon': 2.4,
 'Serene Maiden, Lena': 2.4,
 'Zypsophila Fairy, Asher': 1.2,
 'Protection: Twincast': 0.6,
 'Elementaria Sanctitude': 0.6,
 'Mutual Feelings Maiden, Pense': 0.6,
 'Frenzied Heiress': 0.6,
 'Sincere Confectioner, Viol': 0.4,
 'Deeplands, Gliaane': 0.4,
 'Flowering Sweet Shop': 0.4,
 'Fire Regalis': 0.4,
 'Ghost Chase': 0.4,
 'Champion of Loss': 0.2}

In [15]:
bot_dict

{'Vanilla Trigger': 9.363636363636363,
 'Call of Deep Lands': 4.0,
 'Deeplands, Reguregnus Athanasios': 3.9545454545454546,
 'Deeplands, Orgady': 3.727272727272727,
 'Harvest Elf': 3.727272727272727,
 'Sentinel': 3.227272727272727,
 'Conceited Noble, Philander': 3.1818181818181817,
 'Deeplands, Reguregnus': 3.0,
 'Deeplands, Meydith': 2.9545454545454546,
 'Rotting Usurp Dragon': 2.772727272727273,
 'Frenzied Heiress': 2.227272727272727,
 'Aspiring Maiden, Alana': 2.0454545454545454,
 'Zypsophila Fairy, Asher': 1.4545454545454546,
 'Mutual Feelings Maiden, Pense': 1.0,
 'Protection: Twincast': 0.8181818181818182,
 'Serene Maiden, Lena': 0.8181818181818182,
 'Elementaria Sanctitude': 0.7727272727272727,
 'Sure-kill! Seed Machine Gun!': 0.3181818181818182,
 'Flame Leaf Chaotic Dance, Hakusui&Remred': 0.3181818181818182,
 'Fire Regalis': 0.18181818181818182,
 'Alchemic Hedgehog': 0.09090909090909091,
 'Deeplands, Gliaane': 0.045454545454545456}

In [16]:
all_dict

{'Vanilla Trigger': 9.222222222222221,
 'Call of Deep Lands': 4.0,
 'Deeplands, Reguregnus Athanasios': 3.962962962962963,
 'Harvest Elf': 3.740740740740741,
 'Deeplands, Orgady': 3.6666666666666665,
 'Sentinel': 3.259259259259259,
 'Conceited Noble, Philander': 3.1481481481481484,
 'Deeplands, Reguregnus': 3.0,
 'Deeplands, Meydith': 2.962962962962963,
 'Rotting Usurp Dragon': 2.7037037037037037,
 'Aspiring Maiden, Alana': 2.259259259259259,
 'Frenzied Heiress': 1.9259259259259258,
 'Zypsophila Fairy, Asher': 1.4074074074074074,
 'Serene Maiden, Lena': 1.1111111111111112,
 'Mutual Feelings Maiden, Pense': 0.9259259259259259,
 'Protection: Twincast': 0.7777777777777778,
 'Elementaria Sanctitude': 0.7407407407407407,
 'Sure-kill! Seed Machine Gun!': 0.25925925925925924,
 'Flame Leaf Chaotic Dance, Hakusui&Remred': 0.25925925925925924,
 'Fire Regalis': 0.2222222222222222,
 'Deeplands, Gliaane': 0.1111111111111111,
 'Sincere Confectioner, Viol': 0.07407407407407407,
 'Flowering Sweet Shop

In [23]:
diff_dict = defaultdict(int)
for k, v in all_dict.items():
    try:
        diff_dict[k] = top_dict[k] - v
    except:
        continue

diff_dict = {k:v for k,v in sorted(diff_dict.items(), key=lambda k: k[1], reverse=True)}

In [24]:
diff_dict

{'Serene Maiden, Lena': 1.2888888888888888,
 'Aspiring Maiden, Alana': 0.9407407407407411,
 'Sincere Confectioner, Viol': 0.32592592592592595,
 'Flowering Sweet Shop': 0.32592592592592595,
 'Ghost Chase': 0.32592592592592595,
 'Deeplands, Gliaane': 0.2888888888888889,
 'Fire Regalis': 0.1777777777777778,
 'Champion of Loss': 0.16296296296296298,
 'Sentinel': 0.14074074074074083,
 'Harvest Elf': 0.0592592592592589,
 'Deeplands, Reguregnus Athanasios': 0.0370370370370372,
 'Deeplands, Meydith': 0.0370370370370372,
 'Call of Deep Lands': 0.0,
 'Deeplands, Reguregnus': 0.0,
 'Elementaria Sanctitude': -0.14074074074074072,
 'Conceited Noble, Philander': -0.14814814814814836,
 'Protection: Twincast': -0.1777777777777778,
 'Zypsophila Fairy, Asher': -0.20740740740740748,
 'Deeplands, Orgady': -0.2666666666666666,
 'Rotting Usurp Dragon': -0.3037037037037038,
 'Mutual Feelings Maiden, Pense': -0.32592592592592595,
 'Vanilla Trigger': -0.6222222222222218,
 'Frenzied Heiress': -1.325925925925926

Ah! I noticed an issue where cards not being in both dicts causes an issue. This sorta ruins the tool, 